# Enterprise Email Triage Agent - RL Training

Training a Llama-3-8B model using Unsloth and TRL for the Enterprise Email Triage environment.

## Overview
- **Model**: Llama-3-8B-Instruct with 4-bit quantization
- **Environment**: EnterpriseEmailEnv with LLM tool calling
- **Framework**: TRL (Transformers Reinforcement Learning)
- **Optimization**: LoRA adapters with r=16
- **Target**: Learn optimal email triage decisions

In [ ]:
# Cell 1: Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft torch transformers openenv-core accelerate
!pip install bitsandbytes==0.43.1
!pip install sentencepiece protobuf

In [ ]:
# Cell 2: Model & Tokenizer Setup (Unsloth)
import torch
from unsloth import FastLanguageModel
from peft import LoraConfig, get_peft_model
import json
import re
from transformers import AutoTokenizer

# Load model with 4-bit quantization to prevent Colab OOM
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    load_in_4bit=True,
    device_map="auto"
)

# Configure LoRA adapter
lora_config = LoraConfig(
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Enable training mode
model.train()
print("Model setup complete!")

In [ ]:
# Cell 3: Environment Integration
import sys
import os
from env import EnterpriseEmailEnv
import numpy as np
from typing import Dict, Any

# Initialize environment
env = EnterpriseEmailEnv()
observation = env.reset()
print(f"Environment initialized with {len(env.emails)} emails")

def format_observation_to_prompt(observation: Dict[str, Any]) -> str:
    """
    Convert environment observation to strict system + user prompt for LLM.
    """
    current_email = observation['current_email']
    
    system_prompt = """You are a strict Enterprise Email Triage Agent. You do not explain yourself. You ONLY output a single, valid JSON object.

The JSON must have EXACTLY this structure: {\"tool\": \"<tool_name>\", \"arguments\": {<args>}}

You must choose ONE tool from this exact list: auto_reply, route_to_human, ask_for_clarification.

If you use route_to_human, your arguments MUST include department.

If you use auto_reply, your arguments MUST include message.

All arguments MUST include email_id of the current email.

DO NOT hallucinate. DO NOT invent tools. DO NOT explain your reasoning. ONLY output JSON object."""
    
    user_prompt = f"""Current email requiring triage:

{json.dumps(observation, indent=2)}

Analyze this email and decide the best action. Respond with the appropriate tool call in JSON format."""
    
    # Combine for model input
    full_prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    
    return full_prompt

In [ ]:
# Cell 4: The RL Training Loop (TRL)
from trl import PPOTrainer, PPOConfig
from transformers import TrainingArguments
from datasets import Dataset
import torch.nn.functional as F
from tqdm import tqdm
import time

# Training configuration
training_config = PPOConfig(
    batch_size=4,
    mini_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    max_grad_norm=1.0,
    log_with="tensorboard",
    project_kwargs={"project_name": "email-triage-agent"},
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./email_trainer_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    logging_steps=10,
    save_steps=100,
    num_train_epochs=3,
    fp16=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

# Custom rollout collection for tool-calling
def collect_rollouts(env, model, tokenizer, num_episodes=5):
    """
    Collect rollout data for PPO training.
    """
    rollout_data = []
    
    for episode in tqdm(range(num_episodes), desc="Collecting rollouts"):
        obs = env.reset()
        episode_data = []
        
        while not env.current_step >= env.max_steps and len(env.processed_emails) < len(env.emails):
            # Format observation for model
            prompt = format_observation_to_prompt(obs)
            
            # Generate action from model
            inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    temperature=0.1,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            
            # Extract and parse JSON action
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Extract JSON from generated text
            json_match = re.search(r'\{.*\}', generated_text)
            if json_match:
                try:
                    action = json.loads(json_match.group())
                    # Validate action format
                    if "tool" in action and "arguments" in action:
                        pass  # Valid format
                    else:
                        action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
                except:
                    action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
            else:
                action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
            
            # Take action in environment
            next_obs, reward, done, info = env.step(action)
            
            # Store transition
            episode_data.append({
                "prompt": prompt,
                "action": action,
                "reward": reward,
                "observation": next_obs
            })
            
            # Progress tracking
            print(f"Step {env.current_step}: {action['tool']} -> Reward: {reward:+.2f} | Total: {env.total_reward:+.2f}")
            
            obs = next_obs
            
            if done:
                break
        
        rollout_data.extend(episode_data)
    
    return rollout_data

In [ ]:
# Cell 5: Training Execution
# Collect initial rollouts
print("Collecting initial rollouts...")
rollout_data = collect_rollouts(env, model, tokenizer, num_episodes=3)
print(f"Collected {len(rollout_data)} transitions")

# Convert to dataset format for TRL
def prepare_dataset(rollout_data):
    """Prepare dataset for PPO training."""
    dataset_dict = {
        "prompt": [item["prompt"] for item in rollout_data],
        "chosen": [json.dumps(item["action"]) for item in rollout_data],
        "reward": [item["reward"] for item in rollout_data],
    }
    return Dataset.from_dict(dataset_dict)

# Prepare dataset
train_dataset = prepare_dataset(rollout_data)
print(f"Dataset prepared with {len(train_dataset)} examples")

# Initialize PPO trainer
ppo_trainer = PPOTrainer(
    config=training_config,
    model=model,
    ref_model=None,  # Using same model as reference for simplicity
    tokenizer=tokenizer,
    dataset=train_dataset,
    args=training_args,
)

# Start training
print("Starting PPO training...")
ppo_trainer.train()
print("Training completed!")

In [ ]:
# Cell 6: Save the Adapter
# Create output directory
!mkdir -p ./trained_email_agent

# Save LoRA adapter
adapter_path = "./trained_email_agent/lora_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"LoRA adapter saved to {adapter_path}")
print("Training complete! Model is ready for deployment.")

## Training Summary

This notebook trains a Llama-3-8B model to perform email triage using reinforcement learning:

1. **Model Setup**: 4-bit quantized Llama-3-8B with LoRA (r=16)
2. **Environment Integration**: EnterpriseEmailEnv with JSON tool-calling format
3. **Data Collection**: Rollouts with reward tracking
4. **PPO Training**: Using TRL for policy optimization
5. **Adapter Saving**: LoRA weights for deployment

The trained model learns to:
- Route VIP emergencies to Emergency Support
- Auto-reply to password resets appropriately
- Detect phishing attempts and route to Security
- Handle HR issues with empathy (route to HR)
- Use clarification for ambiguous requests

## Usage

After training, load the model with:
```python
from unsloth import FastLanguageModel
from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    load_in_4bit=True
)
model = PeftModel.from_pretrained(base_model, "./trained_email_agent/lora_adapter")
```